## LES MODULES

In [10]:
%load_ext autoreload
%autoreload 1
import nbimporter
import time
from tqdm import tqdm
import numpy as np
from sklearn.datasets import make_blobs,make_circles
from sklearn.metrics import accuracy_score
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from mpl_toolkits.mplot3d import Axes3D
import tensorflow_datasets as tfds


from IPython.display import HTML
from IPython.display import clear_output
from IPython.display import clear_output, display
import os


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Importe les fichiers:'AtadiaSah','exemple_fonction_indicatrice','Dataset'

In [11]:
import sys
import importlib
module_nom= ['AtadiaSah','exemple_fonction_indicatrice','Dataset']
for i in module_nom:
    if i in sys.modules:
        del sys.modules[i]
import AtadiaSah as As
importlib.reload(As)
import exemple_fonction_indicatrice as FI
importlib.reload(FI)
import Dataset
importlib.reload(Dataset)

<module 'Dataset' from 'Dataset.ipynb'>

## LE MODELE

In [1]:
def RN():
    class modèle_ANN:
        
        def __init__(self):
            self.couches=[]
            self.loss=[]
            self.score=[]
            self.regularisation=[]
            self.optimiseur=[]
            self.modele_fil=[]
            self.modele_save=[]
            self.clear=[]
        def add_couche(self,réseau_plus):
            self.couches=[]
            self.couches.append(réseau_plus)
    
        def add_loss(self,nom_fonction_erreur,epsilon=1e-16):
            self.loss=[nom_fonction_erreur,epsilon]
            
        def add_score(self,nom_methode_evaluation):
            self.score=[nom_methode_evaluation]
            
        def add_regularisation(self,nom_regularisation,alpha=0,beta=0):
            self.regularisation=[nom_regularisation,alpha,beta]
                                    
        def add_optimiseur(self,nom_optimiseur,nature_pas_gradient,pas_gradient=0.01,gamma=0,taille_batch=0):
            self.optimiseur=[nom_optimiseur,nature_pas_gradient,pas_gradient,gamma,taille_batch]
            #else: self.optimiseur=[nom_optimiseur,taille_batch,nature_pas_gradient,pas_gradient,gamma]
            
        def fil(self,modele,nbre_iter=1,pas_save=1,*X):
        
            X_train,y_train,X_testset,y_testset=As.recupe(*X)
            Fonction_modele=As.actualisation(modele)
            paramètres=As.initialisation(Fonction_modele[0],X_train)
        
            coût_trainset,Performance_trainset=[],[]                                                        # sur les données du trainset
            coût_testset,Performance_testset=[],[]                                                        # sur le les doonnées testset
            temps_écouler=[]                                                                       #temps écouler en 10 iteration d'entrainement
            n1=len(paramètres)//2
            temps_initiale=time.time()
            paramètres_evoluant={}
            pas=modele.optimiseur[2]  
    
            bar=tqdm(range(nbre_iter+1),desc='Apprentissage: loss_trainset=0.0,score_trainset=0.0',ncols=130,position=0)
            
            for i in bar:
                sortie_linéaire1,activation1=As.Forword_propagation(X_train,modele.couches[0],paramètres)
                if type(X_testset)!=str:
                    sortie_linéaire2,activation2=As.Forword_propagation(X_testset,modele.couches[0],paramètres)
                n2=i%pas_save
        
                if n2==0:                                                             #enregistre le court chaque 10 intération
                    n3=i//pas_save
                    paramètres_evoluant[f'P{n3}']=paramètres.copy()
                    
                    r=As.regularisation(paramètres,modele.regularisation[0],modele.regularisation[1],modele.regularisation[2])
                    coût_trainset.append(As.evaluation_perte(modele.loss[0],modele.loss[1])(y_train,activation1,r))
                    Performance_trainset.append(As.performance(modele.score[0])(y_train,activation1))  
                    
                    if type(X_testset)!=str:
                        coût_testset.append(As.evaluation_perte(modele.loss[0],modele.loss[1])(y_testset,activation2,r))
                        Performance_testset.append(As.performance(modele.score[0])(y_testset,activation2))        
                    temps_écouler.append(time.time()-temps_initiale)
                    
                bar.set_description(f'Apprentissage:loss_trainset:{coût_trainset[-1]:.9f},score_trainset:{Performance_trainset[-1]:.3f}')
                
                if modele.optimiseur[0]=='DGC':
                    Gradients=As.Back_propagation(modele.couches[0],y_train,paramètres,sortie_linéaire1,activation1,r,modele.loss)
                    paramètres=As.Descente_gradient(paramètres,Gradients,pas)
                    pas=As.PasGradient(modele.optimiseur[1],modele.optimiseur[3])(i,pas)
                elif modele.optimiseur[0]=='mini_batch':
                    mini_batch=As.batch(X_train,y_train,modele.optimiseur[-1])
                    for j in range(len(mini_batch)//2):
                        sortie_linéaire3,activation3=As.Forword_propagation(mini_batch[f'Xbatch{j}'],modele.couches[0],paramètres)
                        Gradients=As.Back_propagation(modele.couches[0],mini_batch[f'Ybatch{j}'],paramètres,sortie_linéaire3,activation3,r,modele.loss)
                        paramètres=As.Descente_gradient(paramètres,Gradients,pas)
                        pas=As.PasGradient(modele.optimiseur[1],modele.optimiseur[3])(i,pas)
                else:
                    Xbatch,Ybatch=As.batch_echantillon(X_train,y_train,modele.optimiseur[-1])
                    sortie_linéaire3,activation3=As.Forword_propagation(Xbatch,modele.couches[0],paramètres)
                    Gradients=As.Back_propagation(modele.couches[0],Ybatch,paramètres,sortie_linéaire3,activation3,r,modele.loss)
                    paramètres=As.Descente_gradient(paramètres,Gradients,pas)
                    pas=As.PasGradient(modele.optimiseur[1],modele.optimiseur[3])(i,pas)
        
            score=As.performance(modele.score[0])(y_train,activation1)
            print(f'le socre du modèle est de :{score:.2f}')
    
            liste1=['paramètres','loss_trainset','loss_testset','score_trainset','score_testset','temps','gradients','SL_trainset','activ_trainset','pas_gradient','entrée_trainset','cible_trainset','nbre_iter']
            liste2=[paramètres_evoluant,coût_trainset,coût_testset,Performance_trainset,Performance_testset,temps_écouler,Gradients,sortie_linéaire1,activation1,pas,X_train,y_train,nbre_iter]
            sortie_modele={i:j for i,j in zip(liste1,liste2)}
            if type(X_testset)!=str: 
                sortie_modele['SL_testset']=sortie_linéaire2
                sortie_modele['activ_testset']=activation2
                sortie_modele['entrée_testset']=X_testset
                sortie_modele['cible_testset']=y_testset
                
            modele.add_modele_save(sortie_modele)
            
            return paramètres_evoluant,coût_trainset,coût_testset,Performance_trainset,Performance_testset,temps_écouler
    
        def add_modele_save(self,save_paramètres_sortie_modele):
            #self.modele_save=[]
            self.modele_save.append(save_paramètres_sortie_modele)
    
        def continue_fil(self,modele,nbre_iter,pas_save):
        
            X_train=modele.modele_save[0]['activ_trainset'][f'A{0}']
            y_train=modele.modele_save[0]['cible_trainset']
            if 'entrée_testset' in modele.modele_save[0]:
                X_testset=modele.modele_save[-1]['entrée_testset']
                y_testset=modele.modele_save[-1]['cible_testset']
            else: X_testset='non'
             
            Fonction_modele=As.actualisation(modele)
            paramètres=modele.modele_save[0]['paramètres']
            paramètres=paramètres[f'P{len(paramètres)-1}']
            nbre_iter=nbre_iter+modele.modele_save[-1]['nbre_iter']
            
            coût_trainset,Performance_trainset=[],[]                                                        
            coût_testset,Performance_testset=[],[]                                                        
            temps_écouler=[]                                                                      
            n1=len(paramètres)//2
            temps_initiale=time.time()
            paramètres_evoluant={}
            pas=modele.modele_save[0]['pas_gradient'] 
    
            bar=tqdm(range(modele.modele_save[-1]['nbre_iter']+1,nbre_iter+1),desc='Apprentissage: loss_trainset=0.0,score_trainset=0.0',ncols=130,position=0)
    
            for i in bar:
                sortie_linéaire1,activation1=As.Forword_propagation(X_train,modele.couches[0],paramètres)
                if type(X_testset)!=str:
                    sortie_linéaire2,activation2=As.Forword_propagation(X_testset,modele.couches[0],paramètres)
                n2=i%pas_save
                
                if n2==0:                                                             #enregistre le court chaque 10 intération
                    n3=i//pas_save
                    paramètres_evoluant[f'P{n3}']=paramètres.copy()
                        
                    r=As.regularisation(paramètres,modele.regularisation[0],modele.regularisation[1],modele.regularisation[2])
                    coût_trainset.append(As.evaluation_perte(modele.loss[0],modele.loss[1])(y_train,activation1,r))
                    Performance_trainset.append(As.performance(modele.score[0])(y_train,activation1))  
                        
                    if type(X_testset)!=str:
                        r=As.regularisation(paramètres,modele.regularisation[0],modele.regularisation[1],modele.regularisation[2])
                        coût_testset.append(As.evaluation_perte(modele.loss[0],modele.loss[1])(y_testset,activation2,r))
                        Performance_testset.append(As.performance(modele.score[0])(y_testset,activation2))        
                    temps_écouler.append(time.time()-temps_initiale+modele.modele_save[0]['temps'][-1])
                    
                    bar.set_description(f'Apprentissage:loss_trainset:{coût_trainset[-1]:.9f},score_trainset:{Performance_trainset[-1]:.3f}')
               
                r=As.regularisation(paramètres,modele.regularisation[0],modele.regularisation[1],modele.regularisation[2])
                if modele.optimiseur[0]=='DGC':
                    Gradients=As.Back_propagation(modele.couches[0],y_train,paramètres,sortie_linéaire1,activation1,r,modele.loss)
                    paramètres=As.Descente_gradient(paramètres,Gradients,pas)
                    pas=As.PasGradient(modele.optimiseur[1],modele.optimiseur[3])(i,pas)
                elif modele.optimiseur[0]=='mini_batch':
                    batch1=As.batch(X_train,y_train,modele.optimiseur[-1])
                    for j in range(len(batch1)//2):
                        sortie_linéaire3,activation3=As.Forword_propagation(batch1[f'Xbatch{j}'],modele.couches[0],paramètres)
                        Gradients=As.Back_propagation(modele.couches[0],batch1[f'Ybatch{j}'],paramètres,sortie_linéaire3,activation3,r,modele.loss)
                        paramètres=As.Descente_gradient(paramètres,Gradients,pas)
                        pas=As.PasGradient(modele.optimiseur[1],modele.optimiseur[3])(i,pas)
                else:
                    Xbatch,Ybatch=As.batch_echantillon(X_train,y_train,modele.optimiseur[-1])
                    sortie_linéaire3,activation3=As.Forword_propagation(Xbatch,modele.couches[0],paramètres)
                    Gradients=As.Back_propagation(modele.couches[0],Ybatch,paramètres,sortie_linéaire3,activation3,r,modele.loss)
                    paramètres=As.Descente_gradient(paramètres,Gradients,pas)
                    pas=As.PasGradient(modele.optimiseur[1],modele.optimiseur[3])(i,pas)
        
        
            score=As.performance(modele.score[0])(y_train,activation1)
            print(f'le socre du modèle est de :{score:.2f}')
            
            liste1=['paramètres','loss_trainset','loss_testset','score_trainset','score_testset','temps','gradients','SL_trainset','activ_trainset','pas_gradient','entrée_trainset','cible_trainset','nbre_iter']
            liste2=[paramètres_evoluant,coût_trainset,coût_testset,Performance_trainset,Performance_testset,temps_écouler,Gradients,sortie_linéaire1,activation1,pas,X_train,y_train,nbre_iter]
            sortie_modele={i:j for i,j in zip(liste1,liste2)}
            if type(X_testset)!=str:
                sortie_modele['SL_testset']=sortie_linéaire2
                sortie_modele['activ_testset']=activation2
                sortie_modele['entrée_testset']=X_testset
                sortie_modele['cible_testset']=y_testset
               
            if pas_save<nbre_iter: modele.add_modele_save(sortie_modele)
            
            return 
        def continue_fil_condition(self,modele,debut,ajouter_nbre_iter_de,precision_stop,pas_save=10):
    
            score=modele.modele_save[0]['score_trainset'][-1]
            for i1 in range(debut): 
                if precision_stop<=(1-score):
                    modele.continue_fil(modele,ajouter_nbre_iter_de,pas_save)
                    score=modele.modele_save[i1]['score_trainset'][-1]
                else: break     
            return
             
        def graphe2D(self,modele,X_fiture='non',precision=1e-1,taille_frontière=1):
        
            X_trainset=modele.modele_save[-1]['entrée_trainset']
            y_trainset=modele.modele_save[-1]['cible_trainset']
            if 'entrée_testset' in modele.modele_save[-1]:
                X_testset=modele.modele_save[-1]['entrée_testset']
                y_testset=modele.modele_save[-1]['cible_testset']
                loss_testset=modele.modele_save[-1]['loss_testset']
                score_testset=modele.modele_save[-1]['score_testset']
            else: X_testset='non'
        
            h=modele.modele_save[-1]['nbre_iter']
            new_dossier=f'images2D_iter_initiale{X_trainset.shape}'
            chemin=os.path.join('images2D',new_dossier)
            if not os.path.exists(chemin):os.makedirs(chemin)
            
            figure=plt.figure(figsize=(6,3))
            paramètres=modele.modele_save[-1]['paramètres']
            nom=list(paramètres)[-1]
            xf,yf,decision =As.frontière(X_trainset,modele.couches[0],paramètres[nom],precision)
            
            temps=modele.modele_save[-1]['temps'][-1]
            temps_text = plt.text(0.1, 1, '', transform=plt.gca().transAxes, fontsize=12, color='red')      # Texte pour le temps réel
            temps_text .set_text(f'Temps réel: {temps:.2f} secondes')
            plt.scatter(X_trainset[0,:],X_trainset[1,:],c=y_trainset,cmap='summer')                                  # on peut remplacer cmap='summer' par alpha=1,s=X[:,1]*10
            
            if type(X_fiture)!=str:plt.scatter(X_fiture[0,:],X_fiture[1,:],c='violet')
            if type(X_testset)!=str:plt.scatter(X_testset[0,:],X_testset[1,:],c='blue')
            plt.colorbar()
            plt.contour(xf,yf,decision,colors='r',alpha=taille_frontière)
            figure.savefig(os.path.join(chemin,f'images_iter{h}.png'))
            return 
            
        def animation2D(self,modele,nom,pas_iter=1,precision=1e-1,X_fiture='non'):
            
            X_trainset=modele.modele_save[-1]['entrée_trainset']
            y_trainset=modele.modele_save[-1]['cible_trainset']
            if 'entrée_testset' in modele.modele_save[-1]:
                X_testset=modele.modele_save[-1]['entrée_testset']
                y_testset=modele.modele_save[-1]['cible_testset']
                loss_testset=modele.modele_save[-1]['loss_testset']
                score_testset=modele.modele_save[-1]['score_testset']
            else: X_testset='non'
            
                
            loss_trainset=modele.modele_save[-1]['loss_trainset']
            score_trainset=modele.modele_save[-1]['score_trainset']
            
            h=modele.modele_save[-1]['nbre_iter']
            new_dossier=f'images2D_ani_iter_initiale{X_trainset.shape}'
            chemin=os.path.join('images_ani2D',new_dossier)
            if not os.path.exists(chemin):os.makedirs(chemin)
            
            if pas_iter==len(loss_trainset):k=pas_iter
            else:k=1
            precision1=precision
            
            print('Evalution de la phase d\'apprentissage  du réseau...')
    
            def classification(k,precision1):
                for i in tqdm(range(k,len(loss_trainset)+1,pas_iter)):
                    
                    temps=modele.modele_save[-1]['temps'][i-1]
                    paramètres=modele.modele_save[-1]['paramètres']
                    nom=list(paramètres)[i-1]
                    
                    xf,yf,decision =As.frontière(X_trainset,modele.couches[0],paramètres[nom],precision1)
                    
                    figure=plt.figure(figsize=(12,4))
                    clear_output(wait=True)
                    
                    plt.subplot(1,3,1)
                    temps_text = plt.text(0.1, 1, '', transform=plt.gca().transAxes, fontsize=12, color='red') # Texte pour le temps réel
                    temps_text .set_text(f'Temps réel: {temps:.2f} secondes')
                    
                    plt.plot(loss_trainset[0:i],'violet',label='erreurs sur TrainSet')
                    if type(X_testset)!=str:plt.plot(loss_testset[0:i],'orange',label='erreurs sur TestSet')    
                    plt.xlabel('nombre d\'itération')
                    
                    plt.grid()
                    plt.legend()
                    plt.tight_layout()
            
                    plt.subplot(1,3,2)
                    temps_text = plt.text(0.1, 1, '', transform=plt.gca().transAxes, fontsize=12, color='red') # Texte pour le temps réel
                    temps_text .set_text(f'Temps réel: {temps:.2f} secondes')
                    
                    plt.plot(score_trainset[0:i],'g',label='score sur TraineSet')
                    if type(X_testset)!=str:plt.plot(score_testset[0:i],'blue',label='score sur TestSet')
                    plt.xlabel('nombre d\'itération')
                    
                    plt.grid()
                    plt.legend()
                    plt.tight_layout()
                
            
                    plt.subplot(1,3,3)
                    temps_text = plt.text(0.1, 1, '', transform=plt.gca().transAxes, fontsize=12, color='red') # Texte pour le temps réel
                    temps_text.set_text(f'Temps réel: {temps:.2f} secondes')
                
                    plt.scatter(X_trainset[0],X_trainset[1],c=y_trainset,cmap='summer') 
                    plt.contour(xf,yf,decision,colors='r')
                    
                    if type(X_testset)!=str:plt.scatter(X_testset[0,:],X_testset[1,:],alpha=0.7,c='blue') #### ajouter c=y_testset
                    if type(X_fiture)!=str:plt.scatter(X_fiture[0,:],X_fiture[1,:],alpha=0.7,c='violet')
                
                    #plt.grid()
                    plt.tight_layout()
                    figure.savefig(os.path.join(chemin,f'images_ani_iter{i}.png'))
                    plt.show()
                    plt.close(figure)
                return 
            def approximation(k,precision1):
                for i in tqdm(range(k,len(loss_trainset)+1,pas_iter)):
                
                    temps=modele.modele_save[-1]['temps'][i-1]
                    paramètres=modele.modele_save[-1]['paramètres']
                    nom=list(paramètres)[i-1]
                    
                    xf,yf,decision =As.frontière(X_trainset,modele.couches[0],paramètres[nom],precision1)
                        
                    figure=plt.figure(figsize=(12,4))
                    clear_output(wait=True)
                    
                    plt.subplot(1,3,1)
                    temps_text = plt.text(0.1, 1, '', transform=plt.gca().transAxes, fontsize=12, color='red') # Texte pour le temps réel
                    temps_text .set_text(f'Temps réel: {temps:.2f} secondes')
                        
                    plt.plot(loss_trainset[0:i],'violet',label='erreurs sur TrainSet')
                    if type(X_testset)!=str:plt.plot(loss_testset[0:i],'orange',label='erreurs sur TestSet')    
                    plt.xlabel('nombre d\'itération')
                        
                    plt.grid()
                    plt.legend()
                    plt.tight_layout()
                
                    plt.subplot(1,3,2)
                    temps_text = plt.text(0.1, 1, '', transform=plt.gca().transAxes, fontsize=12, color='red') # Texte pour le temps réel
                    temps_text .set_text(f'Temps réel: {temps:.2f} secondes')
    
                    plt.plot(score_trainset[0:i],'g',label='score sur TraineSet')
                    if type(X_testset)!=str:plt.plot(score_testset[0:i],'blue',label='score sur TestSet')
                    plt.xlabel('nombre d\'itération')
                        
                    plt.grid()
                    plt.legend()
                    plt.tight_layout()
                    
                
                    plt.subplot(1,3,3)
                    temps_text = plt.text(0.1, 1, '', transform=plt.gca().transAxes, fontsize=12, color='red') # Texte pour le temps réel
                    temps_text.set_text(f'Temps réel: {temps:.2f} secondes')
                    
                    plt.plot(xf,yf,'r')
                    plt.plot(X_trainset[0],y_trainset[0],'--b') 
                        
                    if type(X_testset)!=str:plt.plot(X_testset[0],X_testset[0],c='blue') #### ajouter c=y_testset
                    if type(X_fiture)!=str:plt.plot(X_fiture[0],X_fiture[0],c='violet')
                    
                    plt.grid()
                    plt.tight_layout()
                    figure.savefig(os.path.join(chemin,f'images_ani_iter{i}.png'))
                    plt.show()
                    plt.close(figure)
                return        
            return eval(nom)(k,precision1)
    
        def graphe2D_approximation(self,modele,donnée,X_fiture='non',y_fiture='non',precision='2D',taille_frontière=1):
        
            X_trainset=modele.modele_save[-1]['entrée_trainset']
            y_trainset=modele.modele_save[-1]['cible_trainset']
            if 'entrée_testset' in modele.modele_save[-1]:
                X_testset=modele.modele_save[-1]['entrée_testset']
                y_testset=modele.modele_save[-1]['cible_testset']
            else: X_testset='non'
                
            
            h=modele.modele_save[-1]['nbre_iter']
            new_dossier=f'images2D_iter_initiale{X_trainset.shape}'
            chemin=os.path.join('images2D',new_dossier)
            if not os.path.exists(chemin):os.makedirs(chemin)
                
            figure=plt.figure(figsize=(6,3))
            paramètres=modele.modele_save[-1]['paramètres']
            nom=list(paramètres)[-1]
            if type(X_fiture)!=str:
                activation=As.Forword_propagation(X_fiture,modele.couches[0],paramètres[nom])[1]
                y_fiture0=activation[f'A{len(activation)-1}']
          
            xf,yf,decision =As.frontière(donnée,modele.couches[0],paramètres[nom],precision)
            
            temps=modele.modele_save[-1]['temps'][-1]
            temps_text = plt.text(0.1, 1, '', transform=plt.gca().transAxes, fontsize=12, color='red')      # Texte pour le temps réel
            temps_text .set_text(f'Temps réel: {temps:.2f} secondes')
           
            if type(X_fiture)!=str:
                plt.plot(X_fiture[0],y_fiture0[0],c='g',label='prédiction_fiture')
                plt.plot(X_fiture[0],y_fiture[0],c='orange',label='fiture_trajectoire_exat')
            plt.plot(xf,yf,'r')
            plt.plot(X_trainset[0],y_trainset[0],'--b',label='courbe_exat')      
            if type(X_testset)!=str : plt.plot(X_testset[0],y_testset[0],alpha=1,c='yellow',label='courbe_testset')
            plt.legend()
            #figure.savefig(os.path.join(chemin,f'images_iter{h}.png'))
            return 
            
        def graphe3D(self,modele,taille,taille_test=10,precision='3D'):
        
            X_trainset=modele.modele_save[-1]['entrée_trainset']
            y_trainset=modele.modele_save[-1]['cible_trainset']
            if 'entrée_testset' in modele.modele_save[-1]:
                X_testset=modele.modele_save[-1]['entrée_testset']
                y_testset=modele.modele_save[-1]['cible_testset']
            else: X_testset='non'
            
            X1=X_trainset[0].reshape(taille,taille)
            X2=X_trainset[1].reshape(taille,taille)
            y_x1=y_trainset[0].reshape(taille,taille)
    
            if type(X_testset)!=str :
                X3=X_testset[0].reshape(taille_test,taille_test)
                X4=X_testset[1].reshape(taille_test,taille_test)
                y_x2=y_testset[0].reshape(taille_test,taille_test)
             
            temps=modele.modele_save[-1]['temps'][-1]
            paramètres=modele.modele_save[-1]['paramètres']
            nom=list(paramètres)[-1]
        
            xf,yf,decision =As.frontière(X_trainset,modele.couches[0],paramètres[nom],precision)
            yf=yf.reshape(taille,taille)
            if type(X_testset)!=str :
                xf1,yf1,decision =As.frontière(X_testset,modele.couches[0],paramètres[nom],precision)
                yf1=yf1.reshape(taille_test,taille_test)
            
            fig=plt.figure(figsize=(12,4))
            ax1=fig.add_subplot(1,2,1,projection='3d')
            ax2=fig.add_subplot(1,2,2,projection='3d')
        
            ax1.plot_surface(X1,X2,yf,cmap='viridis',edgecolor='violet',rstride=3,cstride=3)                 #
            if type(X_testset)!=str : ax1.plot_surface(X3,X4,yf1)#,cmap='viridis',edgecolor='red',rstride=3,cstride=3)  
            ax1.set_title(f'Temps exécution Du ANN: {temps:.2f} secondes',color='red',y=1)
            
            ax2.plot_surface(X1,X2,y_x1,cmap='plasma',edgecolor='b',rstride=3,cstride=3)                 #cmap='viridis',,alpha=0,zorder=5,cmap='viridis'rstride=5,cstride=5,cmap='inferno'
            if type(X_testset)!=str : ax2.plot_surface(X3,X4,y_x2)#,cmap='plasma',edgecolor='b',rstride=3,cstride=3) 
            
            #fig.savefig(f'DGS{j}',bbox_inches='tight')
        
            return plt.show()
            
        def loss_score(self,modele):
            n=len(modele.modele_save)
            loss_trainset1,score_trainset1,temps1,loss_testset1,score_testset1=[],[],[],[],[]
            for i in range(n):
                temps1=temps1+modele.modele_save[i]['temps']
                loss_trainset1=loss_trainset1+modele.modele_save[i]['loss_trainset']
                score_trainset1=score_trainset1+modele.modele_save[i]['score_trainset']
                if 'entrée_testset' in modele.modele_save[-1]:
                    loss_testset1=loss_testset1+modele.modele_save[i]['loss_testset']
                    score_testset1=score_testset1+modele.modele_save[i]['score_testset']
            return loss_trainset1,score_trainset1,temps1,loss_testset1,score_testset1
        def Total_paramètres(self,modele):
            n=len(modele.modele_save)
            paramètres=modele.modele_save[0]['paramètres'] 
            for i in range(1,n):
                paramètres.update(modele.modele_save[i]['paramètres'] )
            return paramètres
    
        def animation3D(self,modele,nom,taille,pas_iter=1,X_fiture='non',precision='3D'):
        
            X_trainset=modele.modele_save[-1]['entrée_trainset']
            y_trainset=modele.modele_save[-1]['cible_trainset']
            if 'entrée_testset' in modele.modele_save[-1]:
                X_testset=modele.modele_save[-1]['entrée_testset']
                y_testset=modele.modele_save[-1]['cible_testset']
                loss_testset=modele.modele_save[-1]['loss_testset']
                score_testset=modele.modele_save[-1]['score_testset']
            else: X_testset='non'
                
            X1=X_trainset[0].reshape(taille,taille)
            X2=X_trainset[1].reshape(taille,taille)
            y_x1=y_trainset[0].reshape(taille,taille)
         
            loss_trainset=modele.modele_save[-1]['loss_trainset']
            score_trainset=modele.modele_save[-1]['score_trainset']
                
            new_dossier=f'images2D_ani_iter_initiale{X_trainset.shape}'
            chemin=os.path.join('images_ani2D',new_dossier)
            if not os.path.exists(chemin):os.makedirs(chemin)
                
            if pas_iter==len(loss_trainset):k=pas_iter
            else:k=1
            precision1=precision
                
            print('Evalution de la phase d\'apprentissage  du réseau...')
            def approximation(k,precision1):
                for i in tqdm(range(k,len(loss_trainset)+1,pas_iter)):
                    
                    temps=modele.modele_save[-1]['temps'][i-1]
                    paramètres=modele.modele_save[-1]['paramètres']
                    nom=list(paramètres)[i-1]
                        
                    xf,yf,decision =As.frontière(X_trainset,modele.couches[0],paramètres[nom],precision1)
                    yf=yf.reshape(taille,taille)
                        
                    fig=plt.figure(figsize=(12,4))
                    clear_output(wait=True)
                    ax1=fig.add_subplot(1,3,1)
                    ax1.plot(loss_trainset[0:i],'violet',label='erreurs sur TrainSet')
                    if type(X_testset)!=str:ax1.plot(loss_testset[0:i],'orange',label='erreurs sur TestSet')    
                    plt.xlabel('nombre d\'itération')
                    plt.grid()
                    plt.legend()
                    plt.tight_layout()
        
                    ax2=fig.add_subplot(1,3,2)
                    ax2.plot(score_trainset[0:i],'g',label='score sur TraineSet')
                    if type(X_testset)!=str:ax2.plot(score_testset[0:i],'blue',label='score sur TestSet')
                    plt.xlabel('nombre d\'itération')
                    plt.grid()
                    plt.legend()
                    plt.tight_layout()
                    
                    ax3=fig.add_subplot(1,3,3,projection='3d')
                    ax3.plot_surface(X1,X2,yf,cmap='viridis',edgecolor='violet',rstride=3,cstride=3,label='Approximation \n de f par un ANN') 
                    ax3.plot_surface(X1,X2,y_x1,cmap='plasma',edgecolor='b',rstride=3,cstride=3,alpha=0.5,label='Fonction f')  
                    ax3.set_title(f'Temps exécution Du ANN: {temps:.2f} secondes',color='red',y=0.97)
                    plt.legend(bbox_to_anchor=(1.1,0.93))
                    #figure.savefig(os.path.join(chemin,f'images_ani_iter{i}.png'))
                    
                    plt.show()
                    plt.close(fig)
                return 
            return  eval(nom)(k,precision)

        def evo(self,modele,a0=('non','non'),a=('non','non'),b=('non','non'),precision1=1e-3,pas=0):
            nom_probleme,nom_data=list(a0)[0],list(a0)[1]
            donnée,X_fiture=list(a)[0],list(a)[1]
            taille,taille_test=list(b)[0],list(b)[1]
            figure=plt.figure(figsize=(12,4))
            ax1 = figure.add_subplot(1,3,1)
            ax2 = figure.add_subplot(1,3,2)
            if type(taille)!=str: ax3=figure.add_subplot(1,3,3,projection='3d')
            else: ax3 = figure.add_subplot(1,3,3)
            
            loss_trainset,score_trainset,temps,loss_testset,score_testset=modele.loss_score(modele)
            X_trainset=modele.modele_save[-1]['entrée_trainset']
            y_trainset=modele.modele_save[-1]['cible_trainset']
            if 'entrée_testset' in modele.modele_save[-1]:
                X_testset=modele.modele_save[-1]['entrée_testset']
                y_testset=modele.modele_save[-1]['cible_testset']
            else: X_testset='non'
                
            n=len(loss_trainset)
            n1=len(loss_testset)
            def animate(k):
                ax1.clear()  
                ax2.clear()  
                ax3.clear()  
               
                ax1.set_title(f'Evolution des erreurs',color='red')  #{i/10:.2f}
                ax2.set_title(f'Evolution des performances',color='red') 
                
                ax1.plot(loss_trainset[0:k],'violet',label='Erreurs sur TrainSet') 
                ax2.plot(score_trainset[0:k],'g',label='Score sur TraineSet')
                paramètres1=modele.Total_paramètres(modele)
                if n1>1:
                    ax1.plot(loss_testset[0:k],'orange',label='Erreurs sur TestSet')    
                    ax2.plot(score_testset[0:k],'blue',label='score sur TestSet')
                if X_trainset.shape[0]<3:
                    nom=list(paramètres1)[k-1]
                    
                    if nom_probleme=='classificationDeuxClasse':
                        xf,yf,decision =As.frontière(X_trainset,modele.couches[0],paramètres1[nom],precision1)
                        ax3.scatter(X_trainset[0],X_trainset[1],c=y_trainset,cmap='summer') 
                        ax3.contour(xf,yf,decision,colors='r')      
                        if type(X_testset)!=str:ax3.scatter(X_testset[0,:],X_testset[1,:],alpha=0.7,c='blue') 
                        if type(X_fiture)!=str:ax3.scatter(X_fiture[0,:],nom_data[1,:],alpha=0.7,c='violet')
                        ax3.set_title(f'Frontière de decision en rouge',color='red') 
                            
                    if nom_probleme=='approximation2D':
                        if type(donnée)!=str:xf,yf,decision =As.frontière(donnée,modele.couches[0],paramètres1[nom],'non')
                        ax3.plot(xf,yf,'r',label='Courbe_approchée')
                        ax3.plot(X_trainset[0],y_trainset[0],'--b',label='Courbe_exact')    
                        if type(X_testset)!=str:ax3.plot(X_testset[0],y_testset[0],c='blue',label='Courbe_test') 
                        if type(X_fiture)!=str :
                            activation=As.Forword_propagation(X_fiture,modele.couches[0],paramètres1[nom])[1]
                            y_fiture0=activation[f'A{len(activation)-1}']
                            ax3.plot(X_fiture[0],y_fiture0[0],c='g',label='prédiction_fiture')
                            if type(nom_data)!=str: 
                                ax3.plot(X_fiture[0],nom_data[0],c='orange',label='courbe_fiture_exat')
                        ax3.set_title(f'Approximation par ANN',color='red') 
                        
                    if nom_probleme=='approximation3D':
                        X1=X_trainset[0].reshape(taille,taille)
                        X2=X_trainset[1].reshape(taille,taille)
                        y_x1=y_trainset[0].reshape(taille,taille)
                        if type(X_testset)!=str :
                            X3=X_testset[0].reshape(taille_test,taille_test)
                            X4=X_testset[1].reshape(taille_test,taille_test)
                            y_x2=y_testset[0].reshape(taille_test,taille_test)
                        xf,yf,decision =As.frontière(X_trainset,modele.couches[0],paramètres1[nom],'3D')
                        yf=yf.reshape(taille,taille)
                        if type(X_testset)!=str :
                            xf1,yf1,decision =As.frontière(X_testset,modele.couches[0],paramètres1[nom],'3D')
                            yf1=yf1.reshape(taille_test,taille_test)
                        ax3.plot_surface(X1,X2,yf,cmap='viridis',edgecolor='violet',rstride=2,cstride=2,label='surface_approchée') 
                        ax3.plot_surface(X1,X2,y_x1,cmap='plasma',edgecolor='b',rstride=3,cstride=3,alpha=0.5,label='surface_exact')  
                        if type(X_testset)!=str : ax3.plot_surface(X3,X4,y_x2,label='surface_testset')  
                        ax3.set_title(f'Approximation par un ANN' , color='red')
                        ax3.legend()
                else:
                    if nom_probleme=='classificationMultiClasse':
                        p=np.random.randint(donnée.shape[0])
                        if type(X_fiture)!=str and nom_data=='chats_chiens':
                            p=np.random.randint(X_fiture.shape[1])
                            image=X_fiture[:,p].reshape(X_fiture[:,p].shape[0],1)
                            q,s=modele.prediction(modele,image)
                            ax3.imshow(image.reshape(donnée.shape[1],donnée.shape[2]).T)
                            if q[0,0]==True:resultat=f'Chien à {s[0,0]*100:.2f}% de chance'
                            else:resultat=f'Chat à {(1-s[0,0])*100:.2f}% de chance'
                            ax3.set_xlabel(f'{resultat}',color='green')
                            ax3.set_title(f'DataSet des chats et chien',color='red')
                        elif type(X_fiture)!=str and nom_data=='MNIST':
                            p=np.random.randint(X_fiture.shape[1])
                            image=X_fiture[:,p].reshape(X_fiture[:,p].shape[0],1)
                            nom=list(paramètres1)[-1]
                            paramètres1=paramètres1[nom]
                            n2=len(paramètres1)//2
                            image=image.reshape(image.shape[0],1)
                            activation=As.Forword_propagation(image,modele.couches[0],paramètres1)[1][f'A{n2}']
                            s=np.argmax(activation,axis=0)
                            ax3.imshow(image.reshape(donnée.shape[1],donnée.shape[2]).T) 
                            resultat=f'Le chiffre est {s[0]}: Certitude  de {(activation[s,0][0]*100):.2f}% de chance'
                            ax3.set_xlabel(f'{resultat}',color='green')
                            ax3.set_title(f'DataSet MNIST',color='red')
                        else:
                            ax3.imshow(donnée[p])
                            ax3.set_title(f'DataSet',color='red')
                                 
                ax1.grid()
                ax1.set_xlabel(f'Nombre d\'itération')
                ax1.set_ylabel(f'Erreur')
                ax1.legend()
                
                ax2.grid()
                ax2.set_xlabel(f'Nombre d\'itération')
                ax2.set_ylabel(f'Capacité')
                ax2.legend()
                if type(taille)==str and type(nom_data)!=str:ax3.legend()   #à corriger ....
                
                return 
            bar1=tqdm(np.arange(1, n+1,pas),colour='cyan',position=0)
            ani = animation.FuncAnimation(figure, animate, frames=bar1, interval=20)
            plt.close()
            HTML(ani.to_jshtml())
            return ani,figure
    
    
        
        def prediction(self,modele,X_fiture):
            paramètres=modele.modele_save[-1]['paramètres']
            nom=list(paramètres)[-1]
            paramètres=paramètres[nom]
            P=As.prediction(X_fiture,modele.couches[0],paramètres)
            return P
            
        def predition_chats_chiens(self,X_train1,X_fiture):
            image=X_fiture.reshape(X_fiture.shape[0],1)
            q,s=modele.prediction(image)
            plt.imshow(image.reshape(X_train1.shape[1],X_train1.shape[2]).T)
            if q[0,0]==True:resultat=f'Chien à {(s[0,0]*100):.2f}% de chance'
            else: resultat=f'Chat à {(1-s[0,0])*100:.2f}% de chance'
            return resultat
            
        def prediction_MNIST(self,modele,X_trian1,image):
            paramètres=modele.modele_save[-1]['paramètres']
            nom=list(paramètres)[-1]
            paramètres=paramètres[nom]
            n1=len(paramètres)//2
            image=image.reshape(image.shape[0],1)
            activation=As.Forword_propagation(image,modele.couches[0],paramètres)[1][f'A{n1}']
            p=np.argmax(activation,axis=0)
            plt.imshow(image.reshape(X_trian1.shape[1],X_train1.shape[2]).T) 
            resultat=f'Le chiffre est {p[0]}: Certitude  de {(activation[p,0][0]*100):.2f}% de chance'
            return resultat
            
        def clear(self,modele):
            modele.madele_save=[]
    return modèle_ANN()